In [1]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import os

# 1. Define strict, explicit schemas to handle OULAD data structures safely
student_info_schema = StructType([
    StructField("code_module", StringType(), True),
    StructField("code_presentation", StringType(), True),
    StructField("id_student", IntegerType(), False), # Primary identifier (integer)
    StructField("gender", StringType(), True),
    StructField("region", StringType(), True),
    StructField("highest_education", StringType(), True),
    StructField("imd_band", StringType(), True), # Socio-economic data
    StructField("age_band", StringType(), True),
    StructField("num_of_prev_attempts", IntegerType(), True),
    StructField("studied_credits", IntegerType(), True),
    StructField("disability", StringType(), True), # HCAI protected attribute
    StructField("final_result", StringType(), True) # Target variable
])

student_vle_schema = StructType([
    StructField("code_module", StringType(), True),
    StructField("code_presentation", StringType(), True),
    StructField("id_student", IntegerType(), False),
    StructField("id_site", IntegerType(), True),
    StructField("date", IntegerType(), True), # Asynchronous relative-day offset
    StructField("sum_click", IntegerType(), True) # Continuous interaction count
])

# 2. Source and Target path configurations
SOURCE_DIR = "Files/raw_csv/"
print("[1/3] Ready to convert raw text logs to structured Delta tables...")

try:
    # --- CONVERT STUDENT INFO ---
    print("Processing 'studentInfo.csv' via PySpark engine...")
    df_info = spark.read.format("csv") \
        .option("header", "true") \
        .schema(student_info_schema) \
        .load(f"{SOURCE_DIR}studentInfo.csv")
        
    # Write directly to the Delta Tables layer
    df_info.write.format("delta").mode("overwrite").saveAsTable("studentInfo_bronze")
    print("✔ Successfully saved managed Delta table: studentInfo_bronze")

    # --- CONVERT STUDENT VLE (10.6 Million Rows) ---
    print("Processing massive 'studentVle.csv' clickstream logs (10.6M rows)...")
    df_vle = spark.read.format("csv") \
        .option("header", "true") \
        .schema(student_vle_schema) \
        .load(f"{SOURCE_DIR}studentVle.csv")
        
    # Write clickstream table to Delta tables layer
    df_vle.write.format("delta").mode("overwrite").saveAsTable("studentVle_bronze")
    print("✔ Successfully saved managed Delta table: studentVle_bronze")
    
    print("\n🚀 SUCCESS: Managed tables successfully compiled inside Bronze Layer!")
    print("Fetching active Spark database catalog metrics...")
    
    # Native Spark API: Lists all tables in the active workspace catalog without needing a 'default' database string
    display(spark.catalog.listTables())

except Exception as e:
    print(f"❌ SCHEMA ENFORCEMENT FAILURE ERROR: {str(e)}")


StatementMeta(, e20931b5-51e8-4bfc-8907-d6e6857fbfc9, 3, Finished, Available, Finished, False)

[1/3] Ready to convert raw text logs to structured Delta tables...
Processing 'studentInfo.csv' via PySpark engine...
✔ Successfully saved managed Delta table: studentInfo_bronze
Processing massive 'studentVle.csv' clickstream logs (10.6M rows)...
✔ Successfully saved managed Delta table: studentVle_bronze

🚀 SUCCESS: Tables compiled! Run 'show tables' to audit storage:
❌ SCHEMA ENFORCEMENT FAILURE ERROR: [SCHEMA_NOT_FOUND] The schema `default.OULAD_DENG_HCAI_RESEARCH.OULAD_DENG_Bronze_Raw.oulad_bron_raw` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a catalog, verify the current_schema() output, or qualify the name with the correct catalog.
To tolerate the error on drop use DROP SCHEMA IF EXISTS.
